# ZenGuard UEBA Anomaly Detection Platform

This notebook implements the User and Entity Behavior Analytics (UEBA) component of the ZenGuard Framework using the Isolation Forest algorithm. Following the base paper, it introduces a separate section for **Report Generation** and an **Interactive Dashboard** to test the model dynamically.

### Key Features
- **Synthetic Data Generation**: Maps exactly to the 7 features extracted by ZenGuard SIEM.
- **Automated Report Generation**: Outputs reports and visualizations to `implementation result/`.
- **Interactive SOC Dashboard**: Process manual inputs or raw JSON payloads from the SIEM to predict threats and trigger SOAR playbooks in real-time.

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn ipywidgets

In [ ]:
import os
import numpy as np
import pandas as pd

# Ensure result directory exists as requested
result_dir = "implementation result"
os.makedirs(result_dir, exist_ok=True)

print("Generating Synthetic Data matching ZenGuard Base Paper...")
np.random.seed(42)
n_samples = 15000

# Normal behavior baselines
session_duration = np.random.normal(loc=2.0, scale=1.5, size=n_samples).clip(0.1, 10)
failed_logins = np.random.poisson(lam=0.2, size=n_samples).clip(0, 3)
access_time = np.random.randint(8, 19, size=n_samples) # Business hours mainly
device_trust_score = np.random.normal(loc=0.9, scale=0.1, size=n_samples).clip(0.5, 1.0)
privilege_change_attempted = np.random.choice([0, 1], p=[0.99, 0.01], size=n_samples)
external_connection = np.random.choice([0, 1], p=[0.8, 0.2], size=n_samples)
MFA_bypassed = np.zeros(n_samples)

data = pd.DataFrame({
    'session_duration': session_duration,
    'failed_logins': failed_logins,
    'access_time': access_time,
    'device_trust_score': device_trust_score,
    'privilege_change_attempted': privilege_change_attempted,
    'external_connection': external_connection,
    'MFA_bypassed': MFA_bypassed
})

# Inject anomalies (15% as per paper)
n_anomalies = int(n_samples * 0.15)
anomaly_indices = np.random.choice(n_samples, n_anomalies, replace=False)

data.loc[anomaly_indices, 'failed_logins'] = np.random.randint(4, 15, size=n_anomalies)
data.loc[anomaly_indices, 'access_time'] = np.random.choice([1, 2, 3, 4, 22, 23], size=n_anomalies)
data.loc[anomaly_indices, 'device_trust_score'] = np.random.uniform(0.1, 0.5, size=n_anomalies)
data.loc[anomaly_indices, 'privilege_change_attempted'] = np.random.choice([0, 1], p=[0.2, 0.8], size=n_anomalies)
data.loc[anomaly_indices, 'external_connection'] = 1
data.loc[anomaly_indices, 'MFA_bypassed'] = np.random.choice([0, 1], p=[0.5, 0.5], size=n_anomalies)

csv_path = os.path.join(result_dir, "synthetic_zenguard_logs.csv")
data.to_csv(csv_path, index=False)
print(f"Generated {n_samples} Synthetic Behavioral Logs! Saved to: {csv_path}")

In [ ]:
from sklearn.ensemble import IsolationForest
import joblib

features = ['session_duration', 'failed_logins', 'access_time', 'device_trust_score', 
            'privilege_change_attempted', 'external_connection', 'MFA_bypassed']
X = data[features]

print("Training ZenGuard UEBA Isolation Forest Model...")
ueba_model = IsolationForest(n_estimators=100, contamination=0.15, random_state=42, n_jobs=-1)
ueba_model.fit(X)

model_path = os.path.join(result_dir, "zenguard_ueba_model.pkl")
joblib.dump(ueba_model, model_path)
print(f"Model successfully saved to: {model_path}")

## Report Generation
Generates risk score distributions, anomaly mapping, and a Markdown evaluation report.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Generating ZenGuard Evaluation Report...")
data['anomaly_prediction'] = ueba_model.predict(X)
data['anomaly_score'] = ueba_model.decision_function(X)

# Map predictions: -1 (anomaly) -> Risk 95, 1 (normal) -> Risk 50 (From Paper)
data['risk_score'] = np.where(data['anomaly_prediction'] == -1, 95, 50)

plt.figure(figsize=(10, 5))
sns.histplot(data['risk_score'], bins=10, kde=False, color='#2c3e50')
plt.title('UEBA Risk Score Distribution')
plt.xlabel('Risk Score (50=Normal, 95=High Risk)')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.6)
plot_path = os.path.join(result_dir, 'risk_score_distribution.png')
plt.savefig(plot_path)
plt.close()

plt.figure(figsize=(10, 5))
sns.scatterplot(x='failed_logins', y='anomaly_score', hue='risk_score', 
                palette={50: '#27ae60', 95: '#c0392b'}, data=data, alpha=0.5)
plt.title('Anomaly Score vs Failed Logins')
plt.xlabel('Failed Logins')
plt.ylabel('Isolation Forest Decision Function Score')
plt.grid(True, linestyle='--', alpha=0.6)
scatter_path = os.path.join(result_dir, 'anomaly_scatter.png')
plt.savefig(scatter_path)
plt.close()

report_content = f"""# ZenGuard UEBA Model Evaluation Report\n
## Overview\n- **Total Samples:** {n_samples}\n- **Assumed Contamination Rate:** 15%\n- **High-Risk Events Detected (Risk = 95):** {len(data[data['risk_score'] == 95])}\n- **Normal Events (Risk = 50):** {len(data[data['risk_score'] == 50])}\n
## Model Configuration\n- **Algorithm:** Isolation Forest\n- **Estimators:** 100\n- **Features Analyzed:** {', '.join(features)}\n
## Visualization Artifacts\n- Risk Score Distribution: `risk_score_distribution.png`\n- Anomaly Behavior Scatter Mapping: `anomaly_scatter.png`\n"""
report_path = os.path.join(result_dir, "ueba_evaluation_report.md")
with open(report_path, "w") as f:
    f.write(report_content)
print(f"Reports and plots successfully saved to '{result_dir}' directory.")

## ZenGuard SOC Dashboard
Interactive Dashboard to test the trained model using manual values or SIEM Event structured JSON strings (as parsed by ZenGuard listener component).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
import json

display(HTML("""
<style>
    .dashboard-container { padding: 20px; background-color: #f8f9fa; border-radius: 8px; border: 1px solid #dee2e6; margin-bottom: 20px; }
    .threat-critical { color: #dc3545; font-weight: bold; }
    .threat-normal { color: #28a745; font-weight: bold; }
    .zenguard-title { color: #2c3e50; font-family: sans-serif; }
</style>
"""))

layout = widgets.Layout(width='auto', margin='5px')

session_duration_input = widgets.FloatText(value=1.5, description='Session Duration (hrs):', style={'description_width': 'initial'}, layout=layout)
failed_logins_input = widgets.IntText(value=0, description='Failed Logins:', style={'description_width': 'initial'}, layout=layout)
access_time_input = widgets.IntSlider(value=14, min=0, max=23, description='Access Time (Hour):', style={'description_width': 'initial'}, layout=layout)
device_trust_input = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.05, description='Device Trust Score:', style={'description_width': 'initial'}, layout=layout)
privilege_input = widgets.Dropdown(options=[('No', 0), ('Yes', 1)], value=0, description='Privilege Change Attempt:', style={'description_width': 'initial'}, layout=layout)
external_conn_input = widgets.Dropdown(options=[('No', 0), ('Yes', 1)], value=0, description='External Connection:', style={'description_width': 'initial'}, layout=layout)
mfa_bypassed_input = widgets.Dropdown(options=[('No', 0), ('Yes', 1)], value=0, description='MFA Bypassed:', style={'description_width': 'initial'}, layout=layout)

manual_vbox = widgets.VBox([
    widgets.HTML("<h3 style='margin-top:0'>⚙️ Manual Playbook Parameters</h3><hr>"),
    session_duration_input, failed_logins_input, access_time_input, 
    device_trust_input, privilege_input, external_conn_input, mfa_bypassed_input
], layout=widgets.Layout(width='48%', padding='15px', border='1px solid #ced4da', border_radius='5px', background_color='white'))

siem_json_template = '''{
  "session_duration": 4.5,
  "failed_logins": 5,
  "access_time": 2,
  "device_trust_score": 0.3,
  "privilege_change_attempted": 1,
  "external_connection": 1,
  "MFA_bypassed": 0
}'''

siem_input = widgets.Textarea(
    value='',
    placeholder='Paste Raw SIEM JSON Log here...\n(E.g., containing failed_logins, MFA_bypassed, etc.)',
    description='',
    layout=widgets.Layout(width='100%', height='230px')
)

json_vbox = widgets.VBox([
    widgets.HTML("<h3 style='margin-top:0'>📡 SIEM Event Log (JSON)</h3><p style='font-size:12px; color:#6c757d'><i>If valid JSON is provided here, it overrides manual inputs. Clear this box to use manual sliders.</i></p><hr>"),
    siem_input
], layout=widgets.Layout(width='48%', padding='15px', border='1px solid #ced4da', border_radius='5px', background_color='white'))

input_hbox = widgets.HBox([manual_vbox, json_vbox], layout=widgets.Layout(justify_content='space-between'))

analyze_button = widgets.Button(description="Execute ZenGuard UEBA Analysis", button_style='primary', layout=widgets.Layout(width='100%', height='45px', margin='20px 0', font_weight='bold'))
output_display = widgets.Output()

def perform_analysis(b):
    with output_display:
        output_display.clear_output()
        
        use_json = False
        if siem_input.value.strip():
            try:
                data_dict = json.loads(siem_input.value)
                if all(k in data_dict for k in features):
                    input_df = pd.DataFrame([data_dict])[features]
                    use_json = True
                    print("Status: Parsed SIEM JSON event successfully.")
                else:
                    print("Status: JSON missing required features. Falling back to manual inputs.")
            except json.JSONDecodeError:
                print("Status: Invalid JSON. Falling back to manual inputs.")
                
        if not use_json:
            input_df = pd.DataFrame([{
                'session_duration': session_duration_input.value,
                'failed_logins': failed_logins_input.value,
                'access_time': access_time_input.value,
                'device_trust_score': device_trust_input.value,
                'privilege_change_attempted': privilege_input.value,
                'external_connection': external_conn_input.value,
                'MFA_bypassed': mfa_bypassed_input.value
            }])
            print("Status: Using Manual Parameter Sliders.")
            
        anomaly_pred = ueba_model.predict(input_df)[0]
        raw_score = ueba_model.decision_function(input_df)[0]
        
        risk_score = 95 if anomaly_pred == -1 else 50
        
        if risk_score == 95:
            html_out = f"""
            <div style="padding:15px; border:2px solid #dc3545; border-radius:5px; background-color:#f8d7da; margin-top:10px;">
                <h2 class="threat-critical" style="margin-top:0">🚨 High Risk Detected!</h2>
                <h3 style="margin:10px 0">ZenGuard Risk Score: {risk_score}</h3>
                <p>Isolation Forest Raw Score: {raw_score:.4f}</p>
                <hr style="border-color: #f5c6cb">
                <h4 style="margin-bottom:5px">SOAR Playbook Auto-Responses Triggered:</h4>
                <ul style="margin-top:0">
                    <li><b>Action 1:</b> Enforce Multi-Factor Authentication (MFA)</li>
                    <li><b>Action 2:</b> Isolate Endpoint via EDR</li>
                    <li><b>Action 3:</b> Revoke Privileges & Terminate Session</li>
                </ul>
            </div>
            """
        else:
            html_out = f"""
            <div style="padding:15px; border:2px solid #28a745; border-radius:5px; background-color:#d4edda; margin-top:10px;">
                <h2 class="threat-normal" style="margin-top:0">✅ Normal Behavior</h2>
                <h3 style="margin:10px 0">ZenGuard Risk Score: {risk_score}</h3>
                <p>Isolation Forest Raw Score: {raw_score:.4f}</p>
                <hr style="border-color: #c3e6cb">
                <p style="margin-bottom:0">No anomalous behavior detected. User session proceeds without interruption.</p>
            </div>
            """
        display(HTML(html_out))

analyze_button.on_click(perform_analysis)

main_dashboard = widgets.VBox([
    widgets.HTML("<div class='dashboard-container'><h1 class='zenguard-title' style='margin-top:0'>🛡️ ZenGuard SOC / UEBA Analyst Dashboard</h1><p style='margin-bottom:0'>Test the Isolation Forest model using manual inputs or simulate a SIEM log integration.</p></div>"),
    input_hbox,
    analyze_button,
    output_display
])

display(main_dashboard)
